# Agent Failure Modes — Drift, Loops, Cost Overruns

Production agents fail in ways unit tests rarely catch:

| Failure mode | What goes wrong | Typical signal |
|--------------|-----------------|----------------|
| **Drift** | Quality/path changes over time | Citation rate ↓, new tool sequences |
| **Loops** | Agent repeats tools without progress | Same span name N times |
| **Cost overruns** | Tokens/$ exceed budget | Trace cost >> baseline |

```
  Healthy trace                    Failure traces
  route → tool → answer            route → tool → tool → tool → … (loop)
  pass_rate 0.95                   pass_rate 0.72 (drift)
  $0.003 / ticket                  $0.18 / ticket (cost)
```

This notebook **simulates** each failure on **ShopCo Support Hub** and **ReleaseFlow**, builds **detectors and guards**, and defines **alert + remediation** playbooks — plain Python.

In [ ]:
"""
Agent failure modes — drift, loops, cost overruns.
"""

import json
import os
import random
import re
import uuid
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False
random.seed(7)

MAX_AGENT_STEPS = 6
MAX_COST_USD = 0.05
TOKEN_COST_PER_1K = 0.002


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


POLICY_SNIPPETS = {
    "returns": "Returns within 30 days with receipt. [pol-returns-01]",
    "shipping": "Free shipping over $50. [pol-shipping-02]",
}

ORDERS_DB = {
    "ORD-1001": {"order_id": "ORD-1001", "status": "shipped"},
    "ORD-1002": {"order_id": "ORD-1002", "status": "processing"},
}

print("Failure modes lab ready.")

---

## 1. Trace Model for Failure Detection

In [ ]:
class SpanKind(str, Enum):
    LLM = "llm"
    TOOL = "tool"
    ROUTER = "router"


@dataclass
class Span:
    name: str
    kind: SpanKind
    tokens: int = 0


@dataclass
class AgentRunTrace:
    trace_id: str
    service: str
    user_message: str
    spans: list[Span] = field(default_factory=list)
    final_answer: str = ""
    terminated_reason: str = "complete"

    def path_fingerprint(self) -> str:
        return "→".join(s.name for s in self.spans)

    def total_tokens(self) -> int:
        return sum(s.tokens for s in self.spans)

    def cost_usd(self) -> float:
        return round(self.total_tokens() / 1000 * TOKEN_COST_PER_1K, 4)

    def has_policy_citation(self) -> bool:
        return "[pol-" in self.final_answer


print("Trace model ready.")

---

## 2. Drift — Baseline vs Production Window

In [ ]:
@dataclass
class QualityBaseline:
    service: str
    pass_rate: float
    citation_rate: float
    avg_tool_calls: float
    avg_cost_usd: float
    top_paths: list[str]


@dataclass
class DriftReport:
    drift_detected: bool
    signals: list[str]
    current: dict[str, float]
    baseline: dict[str, float]


SHOPCO_BASELINE = QualityBaseline(
    service="shopco-support",
    pass_rate=0.95,
    citation_rate=0.92,
    avg_tool_calls=2.0,
    avg_cost_usd=0.004,
    top_paths=["route_intent→get_order→search_policy→compose_answer"],
)


def simulate_traces_healthy(n: int) -> list[AgentRunTrace]:
    traces = []
    for _ in range(n):
        t = AgentRunTrace(f"tr-{uuid.uuid4().hex[:6]}", "shopco", "return ORD-1001")
        t.spans = [
            Span("route_intent", SpanKind.ROUTER, 100),
            Span("get_order", SpanKind.TOOL, 0),
            Span("search_policy", SpanKind.TOOL, 0),
            Span("compose_answer", SpanKind.LLM, 150),
        ]
        t.final_answer = f"Shipped. {POLICY_SNIPPETS['returns']}"
        traces.append(t)
    return traces


def simulate_traces_drifted(n: int) -> list[AgentRunTrace]:
    traces = []
    for _ in range(n):
        t = AgentRunTrace(f"tr-{uuid.uuid4().hex[:6]}", "shopco", "return ORD-1001")
        t.spans = [
            Span("route_intent", SpanKind.ROUTER, 100),
            Span("get_order", SpanKind.TOOL, 0),
            Span("compose_answer", SpanKind.LLM, 150),
        ]
        t.final_answer = "You can probably return it."  # drift: no citation
        traces.append(t)
    return traces


def window_metrics(traces: list[AgentRunTrace]) -> dict[str, float]:
    n = max(len(traces), 1)
    cite = sum(1 for t in traces if t.has_policy_citation()) / n
    tools = sum(sum(1 for s in t.spans if s.kind == SpanKind.TOOL) for t in traces) / n
    cost = sum(t.cost_usd() for t in traces) / n
    return {"citation_rate": round(cite, 2), "avg_tool_calls": round(tools, 2), "avg_cost_usd": round(cost, 4)}


print("Healthy:", window_metrics(simulate_traces_healthy(20)))
print("Drifted:", window_metrics(simulate_traces_drifted(20)))

---

## 3. Drift Detector

In [ ]:
def detect_drift(baseline: QualityBaseline, traces: list[AgentRunTrace], cite_tol: float = 0.1) -> DriftReport:
    current = window_metrics(traces)
    signals: list[str] = []
    if current["citation_rate"] < baseline.citation_rate - cite_tol:
        signals.append(f"citation_rate dropped to {current['citation_rate']}")
    paths = Counter(t.path_fingerprint() for t in traces)
    novel = [p for p in paths if p not in baseline.top_paths]
    if novel and paths[novel[0]] / len(traces) > 0.3:
        signals.append(f"novel path dominant: {novel[0]}")
    if current["avg_cost_usd"] > baseline.avg_cost_usd * 2:
        signals.append(f"cost doubled: {current['avg_cost_usd']}")
    return DriftReport(bool(signals), signals, current, {
        "citation_rate": baseline.citation_rate,
        "avg_cost_usd": baseline.avg_cost_usd,
    })


drift_report = detect_drift(SHOPCO_BASELINE, simulate_traces_drifted(30))
print(pretty(drift_report))

---

## 4. Tool Loops — Runaway ReAct

In [ ]:
def run_looping_agent(message: str, max_steps: int = 12) -> AgentRunTrace:
    """Simulates agent that keeps calling search_policy without terminating."""
    trace = AgentRunTrace(f"tr-loop-{uuid.uuid4().hex[:6]}", "shopco", message)
    trace.spans.append(Span("route_intent", SpanKind.ROUTER, 80))
    for i in range(max_steps):
        trace.spans.append(Span("search_policy", SpanKind.TOOL, 200))
        # Never satisfies stop condition
    trace.terminated_reason = "max_steps_forced"
    trace.final_answer = "(timeout)"
    return trace


loop_trace = run_looping_agent("return policy?", max_steps=8)
print("Tool calls:", sum(1 for s in loop_trace.spans if s.name == "search_policy"))

---

## 5. Loop Detector and Circuit Breaker

In [ ]:
@dataclass
class LoopAlert:
    detected: bool
    tool_name: str | None
    repeat_count: int
    action: str


def detect_tool_loop(trace: AgentRunTrace, threshold: int = 3) -> LoopAlert:
    tool_names = [s.name for s in trace.spans if s.kind == SpanKind.TOOL]
    counts = Counter(tool_names)
    for name, cnt in counts.items():
        if cnt >= threshold:
            return LoopAlert(True, name, cnt, "circuit_break: stop agent")
    return LoopAlert(False, None, 0, "continue")


class AgentCircuitBreaker:
    def __init__(self, max_steps: int = MAX_AGENT_STEPS, max_cost: float = MAX_COST_USD) -> None:
        self.max_steps = max_steps
        self.max_cost = max_cost
        self.tripped: list[str] = []

    def check(self, trace: AgentRunTrace) -> tuple[bool, str]:
        if len(trace.spans) >= self.max_steps:
            self.tripped.append("max_steps")
            return False, f"max_steps {self.max_steps} exceeded"
        loop = detect_tool_loop(trace, threshold=3)
        if loop.detected:
            self.tripped.append("tool_loop")
            return False, f"loop on {loop.tool_name} x{loop.repeat_count}"
        if trace.cost_usd() > self.max_cost:
            self.tripped.append("cost_budget")
            return False, f"cost ${trace.cost_usd()} > ${self.max_cost}"
        return True, "ok"


breaker = AgentCircuitBreaker()
ok, reason = breaker.check(loop_trace)
print("Allow run:", ok, "reason:", reason)

---

## 6. Guarded Agent Loop — Stop Early

In [ ]:
def guarded_react(message: str, breaker: AgentCircuitBreaker) -> AgentRunTrace:
    trace = AgentRunTrace(f"tr-guard-{uuid.uuid4().hex[:6]}", "shopco", message)
    trace.spans.append(Span("route_intent", SpanKind.ROUTER, 90))
    for step in range(breaker.max_steps):
        trace.spans.append(Span("search_policy", SpanKind.TOOL, 250))
        allowed, reason = breaker.check(trace)
        if not allowed:
            trace.terminated_reason = reason
            trace.final_answer = f"Stopped: {reason}"
            return trace
        if "[pol-" in POLICY_SNIPPETS["returns"] and step >= 1:
            trace.final_answer = POLICY_SNIPPETS["returns"]
            trace.terminated_reason = "success"
            return trace
    trace.terminated_reason = "max_steps"
    return trace


guarded = guarded_react("returns?", AgentCircuitBreaker(max_steps=6))
print("Steps:", len(guarded.spans), "reason:", guarded.terminated_reason)

---

## 7. Cost Overruns — Token Budget Exhaustion

In [ ]:
def run_expensive_agent(n_llm_calls: int) -> AgentRunTrace:
    trace = AgentRunTrace(f"tr-cost-{uuid.uuid4().hex[:6]}", "shopco", "complex query")
    for i in range(n_llm_calls):
        trace.spans.append(Span(f"llm_refine_{i}", SpanKind.LLM, 800))
    trace.final_answer = "Over-refined answer."
    return trace


expensive = run_expensive_agent(15)
print(f"Tokens: {expensive.total_tokens()}, cost: ${expensive.cost_usd()}")
print(breaker.check(expensive))

---

## 8. ReleaseFlow — Retry Loop on Failed Check

In [ ]:
def release_retry_storm(failures: int = 5) -> AgentRunTrace:
    trace = AgentRunTrace(f"tr-rf-{uuid.uuid4().hex[:6]}", "releaseflow", "deploy 2.4.0")
    for i in range(failures):
        trace.spans.append(Span("run_security_scan", SpanKind.TOOL, 50))
        trace.spans.append(Span("llm_analyze_failure", SpanKind.LLM, 400))
    trace.terminated_reason = "retry_storm"
    return trace


rf_trace = release_retry_storm(6)
rf_breaker = AgentCircuitBreaker(max_steps=8, max_cost=0.02)
print(detect_tool_loop(rf_trace))
print(rf_breaker.check(rf_trace))

---

## 9. Unified Failure Scanner

In [ ]:
@dataclass
class FailureScan:
    trace_id: str
    drift: DriftReport | None
    loop: LoopAlert
    breaker_ok: bool
    breaker_reason: str
    severity: str


def scan_trace(trace: AgentRunTrace, baseline: QualityBaseline, breaker: AgentCircuitBreaker) -> FailureScan:
    drift = detect_drift(baseline, [trace])
    loop = detect_tool_loop(trace)
    ok, reason = breaker.check(trace)
    severity = "low"
    if not ok or loop.detected or drift.drift_detected:
        severity = "high" if not ok and trace.cost_usd() > baseline.avg_cost_usd * 5 else "medium"
    return FailureScan(trace.trace_id, drift if drift.drift_detected else None, loop, ok, reason, severity)


for t in [loop_trace, expensive, simulate_traces_drifted(1)[0]]:
    scan = scan_trace(t, SHOPCO_BASELINE, AgentCircuitBreaker())
    print(t.trace_id, scan.severity, scan.breaker_reason)

---

## 10. Alert Rules

In [ ]:
ALERT_RULES = [
    {"id": "DRIFT-01", "condition": "citation_rate < baseline - 0.1 over 1h", "severity": "warning"},
    {"id": "LOOP-01", "condition": "same tool >= 3 times in one trace", "severity": "critical"},
    {"id": "COST-01", "condition": "trace cost > 5x baseline avg", "severity": "warning"},
    {"id": "COST-02", "condition": "trace cost > hard cap $0.05", "severity": "critical"},
    {"id": "STEP-01", "condition": "spans > MAX_AGENT_STEPS", "severity": "critical"},
]

for rule in ALERT_RULES:
    print(f"{rule['id']:10} {rule['severity']:8} {rule['condition']}")

---

## 11. Remediation Playbook

In [ ]:
REMEDIATION = {
    "drift": [
        "Compare prompt/version deploy timeline",
        "Run offline eval suite against last green baseline",
        "Rollback model or prompt if pass_rate dropped",
    ],
    "loop": [
        "Enable max_steps circuit breaker",
        "Improve tool observation HINT fields",
        "Add done predicate when policy citation present",
    ],
    "cost": [
        "Lower max_tokens per step",
        "Enforce per-trace USD cap",
        "Replace multi-LLM refine with single-shot routing",
    ],
}

for mode, steps in REMEDIATION.items():
    print(f"\n{mode.upper()}:")
    for s in steps:
        print(f"  - {s}")

---

## 12. Production Guards Summary

In [ ]:
@dataclass
class GuardConfig:
    max_steps: int = 6
    max_cost_usd: float = 0.05
    loop_threshold: int = 3
    drift_citation_delta: float = 0.1
    enable_breaker: bool = True


GUARDS = GuardConfig()
print(pretty(GUARDS))

---

## 13. Optional Live LLM — Cost of Extra Refinement

Set `USE_LIVE_LLM = True` to compare one-shot vs two refinement calls (token estimate).

In [ ]:
def estimate_live_cost(rounds: int = 1) -> dict[str, Any]:
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    draft = "Returns in 30 days."
    tokens = 0
    for i in range(rounds):
        resp = llm.invoke([
            SystemMessage(content="Improve this support line; keep citations."),
            HumanMessage(content=draft),
        ])
        draft = str(resp.content)
        tokens += 300  # illustrative per round
    return {"rounds": rounds, "tokens_est": tokens, "cost_usd": round(tokens / 1000 * TOKEN_COST_PER_1K, 4)}


if USE_LIVE_LLM:
    print(estimate_live_cost(1))
    print(estimate_live_cost(5))
else:
    print("Offline: 15 refine LLM calls →", run_expensive_agent(15).cost_usd(), "USD")

---

## 14. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| No baseline | Can't detect drift | Store offline eval metrics |
| No step cap | Infinite loops | `MAX_AGENT_STEPS` |
| Cost alerts on monthly bill only | Too late | Per-trace budget |
| Same threshold for FAQ and ReAct | False alerts | Per-pattern guards |
| Breaker without user message | Confusing UX | Return `terminated_reason` |

---

## 15. Quiz

1. What three failure modes does this notebook focus on?
2. How is drift detected for ShopCo?
3. What trace pattern indicates a tool loop?
4. What does `AgentCircuitBreaker` check?
5. First remediation step for citation rate drift?

---

## 16. Summary

**Drift**, **loops**, and **cost overruns** are agent-specific production failures. Detect them with **baselines**, **path fingerprints**, **loop counters**, and **per-trace budgets**.

ShopCo demos showed citation drift and `search_policy` loops. ReleaseFlow showed retry storms. `AgentCircuitBreaker` unifies guards.

Observe → alert → remediate. Guards belong in the agent runtime, not only in dashboards.